In [ ]:
## 7)(TODO: Did not work as desired.  OPTIONAL: Complicated process) Using Machine Learning Models to Predict Missing Values
Method: Train a machine learning model (Random Forest, XGBoost) to predict missing values based on existing ones.

Pros:
- Highly accurate when patterns exist in the data.
- Works well for both numerical and categorical missing values.

Cons:
- Computationally expensive.
- Requires a significant amount of non-missing data for training.
- Can introduce bias if the model is not well-tuned.

**When to Use?**
- When a dataset is large enough to train a predictive model.
- When missing values depend on multiple variables.


In [ ]:
df = pd.read_csv('missing_synthetic_health_data_500.csv')
print(df)

In [ ]:
#check for missing values

print(df.isnull().sum())

In [ ]:
# Lets compute BMI first

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Step 1: Drop Patient_ID (not useful for prediction)
df_model = df.drop(columns=['Patient_ID'])

# Step 2: Split the data into two sets — one with BMI, one without
df_bmi_known = df_model[df_model['BMI'].notnull()]
df_bmi_missing = df_model[df_model['BMI'].isnull()]

# Step 3: Define features and target
X = df_bmi_known.drop(columns=['BMI'])
y = df_bmi_known['BMI']

# Step 4: Preprocessing — encode categorical variables
categorical_cols = ['Gender', 'Smoker', 'Diabetic', 'Heart_Disease']
numerical_cols = ['Age', 'Blood_Pressure', 'Cholesterol', 'Glucose']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

# Step 5: Create pipeline with RandomForestRegressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Step 6: Fit the model
pipeline.fit(X, y)

# Step 7: Predict missing BMI values
X_missing = df_bmi_missing.drop(columns=['BMI'])
bmi_predicted = pipeline.predict(X_missing)

# Step 8: Fill missing BMI values in the original DataFrame
df.loc[df['BMI'].isnull(), 'BMI'] = bmi_predicted

# Step 9: Verify no more missing BMI values
print(df['BMI'].isnull().sum())  # Should print 0


sns.histplot(df['BMI'], kde=True, color='purple', bins=20)
plt.title("BMI Distribution After Imputation")
plt.xlabel("BMI")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Reload the original data to get BMI with missing values
# (Assuming you've saved the original DataFrame before imputation)
# Otherwise, simulate it again for teaching purpose
df_original = df.copy()
df_original['BMI'] = np.where(df_original.index.isin(df[df['BMI'].isnull()].index), np.nan, df_original['BMI'])

# Step 2: Create two Series for comparison
bmi_before = df_original['BMI'].dropna()   # Only non-missing values
bmi_after = df['BMI']                      # All values (after filling)

# Step 3: Plot KDE or histograms
plt.figure(figsize=(12, 5))

# Histogram comparison
plt.subplot(1, 2, 1)
sns.histplot(bmi_before, kde=True, color='skyblue', label='Before Imputation', bins=20)
sns.histplot(bmi_after, kde=True, color='orange', label='After Imputation', bins=20, alpha=0.6)
plt.title('Histogram: BMI Distribution Before vs After Imputation')
plt.xlabel('BMI')
plt.ylabel('Frequency')
plt.legend()

# KDE comparison
plt.subplot(1, 2, 2)
sns.kdeplot(bmi_before, color='skyblue', label='Before Imputation')
sns.kdeplot(bmi_after, color='orange', label='After Imputation')
plt.title('KDE: BMI Distribution Before vs After Imputation')
plt.xlabel('BMI')
plt.legend()

plt.tight_layout()
plt.show()
